In [ ]:
# =========================================================
# 셀 1. 기본 import & 설정
# =========================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from mpl_toolkits.mplot3d import Axes3D  # 3D 플롯을 위해 필요할 수 있음

plt.rcParams["figure.figsize"] = (6, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 10

In [ ]:
# =========================================================
# 셀 2. 경로 및 출력 폴더 설정
# =========================================================
# 노트북(.ipynb) 파일이 있는 디렉토리 기준
base_dir = Path().resolve()

data_dir = base_dir / "data" / "circle"

# 출력 디렉토리
out_root = base_dir / "outputs" / "circle"
out_individual = out_root / "individual"   # 1~16 개별 궤적 그림
out_group = out_root / "group"             # 그룹 비교 그림
out_features = out_root / "features"       # feature 표 & 통계 그림

for p in [out_individual, out_group, out_features]:
    p.mkdir(parents=True, exist_ok=True)

print("Data dir:", data_dir)
print("Output root:", out_root)

In [ ]:
# =========================================================
# 셀 2. 경로 및 출력 폴더 설정
# =========================================================
# 노트북(.ipynb) 파일이 있는 디렉토리 기준
base_dir = Path().resolve()

data_dir = base_dir / "data" / "circle"

# 출력 디렉토리
out_root = base_dir / "outputs" / "circle"
out_individual = out_root / "individual"   # 1~16 개별 궤적 그림
out_group = out_root / "group"             # 그룹 비교 그림
out_features = out_root / "features"       # feature 표 & 통계 그림

for p in [out_individual, out_group, out_features]:
    p.mkdir(parents=True, exist_ok=True)

print("Data dir:", data_dir)
print("Output root:", out_root)

In [ ]:
# =========================================================
# 셀 3. 궤적 로딩 함수 (1.txt ~ 16.txt)
# =========================================================
def load_trajectory(idx: int):
    """
    idx: 1~16 정수
    반환: numpy 배열 xs, ys, zs
    """
    file_path = data_dir / f"{idx}.txt"
    xs, ys, zs = [], [], []

    with open(file_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            cols = line.split(",")
            if len(cols) <= 6:
                continue
            xyz_str = cols[6].strip()
            parts = xyz_str.split("/")
            if len(parts) != 3:
                continue
            try:
                x, y, z = map(float, parts)
            except ValueError:
                # 숫자로 변환 안되면 스킵
                continue
            xs.append(x)
            ys.append(y)
            zs.append(z)

    xs = np.array(xs, dtype=float)
    ys = np.array(ys, dtype=float)
    zs = np.array(zs, dtype=float)

    return xs, ys, zs

# 테스트용: 하나만 불러보기 (예: 1번)
# xs_test, ys_test, zs_test = load_trajectory(1)
# len(xs_test), len(ys_test), len(zs_test)

In [ ]:
# =========================================================
# 셀 4. 단일 궤적 시각화 유틸 함수 (3D & 투영)
# =========================================================
def get_limits(xs, ys, zs, margin=10.0):
    """단일 궤적에 대한 x/y/z 범위를 margin 포함해서 리턴."""
    xmin, xmax = xs.min(), xs.max()
    ymin, ymax = ys.min(), ys.max()
    zmin, zmax = zs.min(), zs.max()
    return (
        xmin - margin, xmax + margin,
        ymin - margin, ymax + margin,
        zmin - margin, zmax + margin,
    )

def plot_3d(xs, ys, zs, title, save_path):
    fig = plt.figure()
    ax = fig.add_subplot(111, projection="3d")
    ax.set_title(title)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")

    xmin, xmax, ymin, ymax, zmin, zmax = get_limits(xs, ys, zs)
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_zlim(zmin, zmax)

    ax.plot(xs, ys, zs, "-o", markersize=2)

    fig.tight_layout()
    fig.savefig(save_path, dpi=200)
    plt.close(fig)

def plot_projection(u, v, xlabel, ylabel, title, save_path, margin=10.0):
    fig, ax = plt.subplots()
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

    umin, umax = u.min(), u.max()
    vmin, vmax = v.min(), v.max()
    ax.set_xlim(umin - margin, umax + margin)
    ax.set_ylim(vmin - margin, vmax + margin)

    ax.plot(u, v, "-o", markersize=2)
    ax.set_aspect("equal", adjustable="box")

    fig.tight_layout()
    fig.savefig(save_path, dpi=200)
    plt.close(fig)

In [ ]:
# =========================================================
# 셀 5. [1단계] 1~16 개별 궤적 3D + XY/YZ/XZ 저장
# =========================================================
for idx in range(1, 17):
    xs, ys, zs = load_trajectory(idx)

    # 3D
    save3d = out_individual / f"circle_{idx:02d}_3d.png"
    plot_3d(xs, ys, zs, title=f"Trajectory {idx} - 3D", save_path=save3d)

    # XY
    savexy = out_individual / f"circle_{idx:02d}_xy.png"
    plot_projection(xs, ys, xlabel="X", ylabel="Y",
                    title=f"Trajectory {idx} - XY view",
                    save_path=savexy)

    # YZ
    saveyz = out_individual / f"circle_{idx:02d}_yz.png"
    plot_projection(ys, zs, xlabel="Y", ylabel="Z",
                    title=f"Trajectory {idx} - YZ view",
                    save_path=saveyz)

    # XZ
    savexz = out_individual / f"circle_{idx:02d}_xz.png"
    plot_projection(xs, zs, xlabel="X", ylabel="Z",
                    title=f"Trajectory {idx} - XZ view",
                    save_path=savexz)

print("개별 궤적 그림 저장 완료.")

In [ ]:
# =========================================================
# 셀 6. 그룹 데이터 준비 (clean: 1~8, noisy: 9~16)
# =========================================================
group_clean_ids = list(range(1, 9))   # 1~8
group_noisy_ids = list(range(9, 17))  # 9~16

# 미리 메모리에 올리고 싶으면 이렇게:
traj_dict = {}
for idx in range(1, 17):
    xs, ys, zs = load_trajectory(idx)
    traj_dict[idx] = (xs, ys, zs)

print("총 궤적 개수:", len(traj_dict))

In [ ]:
# =========================================================
# 셀 7. [2단계] 그룹 비교 - 3D 오버레이
# =========================================================
def plot_group_3d(traj_dict, clean_ids, noisy_ids, title, save_path, margin=10.0):
    # 두 그룹 전체 범위 계산
    all_x, all_y, all_z = [], [], []
    for idx in clean_ids + noisy_ids:
        xs, ys, zs = traj_dict[idx]
        all_x.append(xs)
        all_y.append(ys)
        all_z.append(zs)
    all_x = np.concatenate(all_x)
    all_y = np.concatenate(all_y)
    all_z = np.concatenate(all_z)

    xmin, xmax = all_x.min() - margin, all_x.max() + margin
    ymin, ymax = all_y.min() - margin, all_y.max() + margin
    zmin, zmax = all_z.min() - margin, all_z.max() + margin

    fig = plt.figure()
    ax = fig.add_subplot(111, projection="3d")
    ax.set_title(title)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_zlim(zmin, zmax)

    # clean 그룹
    for idx in clean_ids:
        xs, ys, zs = traj_dict[idx]
        ax.plot(xs, ys, zs, alpha=0.6, linewidth=1, label="clean" if idx == clean_ids[0] else "")

    # noisy 그룹
    for idx in noisy_ids:
        xs, ys, zs = traj_dict[idx]
        ax.plot(xs, ys, zs, alpha=0.6, linewidth=1, linestyle="--",
                label="noisy" if idx == noisy_ids[0] else "")

    ax.legend()
    fig.tight_layout()
    fig.savefig(save_path, dpi=200)
    plt.close(fig)

save_path_3d_group = out_group / "group_clean_vs_noisy_3d.png"
plot_group_3d(traj_dict, group_clean_ids, group_noisy_ids,title="Clean (1-8) vs Noisy (9-16) - 3D",save_path=save_path_3d_group)

print("그룹 3D 비교 그림 저장 완료.")

In [ ]:
# =========================================================
# 셀 8. [2단계] 그룹 비교 - XY / YZ / XZ 오버레이
# =========================================================
def plot_group_projection(traj_dict, clean_ids, noisy_ids,plane="xy", title="", save_path=None, margin=10.0):
    # plane에 따라 축 선택
    all_u, all_v = [], []

    def get_uv(xs, ys, zs):
        if plane == "xy":
            return xs, ys
        elif plane == "yz":
            return ys, zs
        elif plane == "xz":
            return xs, zs
        else:
            raise ValueError("plane must be one of ['xy', 'yz', 'xz']")

    for idx in clean_ids + noisy_ids:
        xs, ys, zs = traj_dict[idx]
        u, v = get_uv(xs, ys, zs)
        all_u.append(u)
        all_v.append(v)

    all_u = np.concatenate(all_u)
    all_v = np.concatenate(all_v)

    umin, umax = all_u.min() - margin, all_u.max() + margin
    vmin, vmax = all_v.min() - margin, all_v.max() + margin

    fig, ax = plt.subplots()
    ax.set_title(title)

    if plane == "xy":
        ax.set_xlabel("X")
        ax.set_ylabel("Y")
    elif plane == "yz":
        ax.set_xlabel("Y")
        ax.set_ylabel("Z")
    elif plane == "xz":
        ax.set_xlabel("X")
        ax.set_ylabel("Z")

    ax.set_xlim(umin, umax)
    ax.set_ylim(vmin, vmax)

    # clean
    for idx in clean_ids:
        xs, ys, zs = traj_dict[idx]
        u, v = get_uv(xs, ys, zs)
        ax.plot(u, v, alpha=0.6, linewidth=1, label="clean" if idx == clean_ids[0] else "")

    # noisy
    for idx in noisy_ids:
        xs, ys, zs = traj_dict[idx]
        u, v = get_uv(xs, ys, zs)
        ax.plot(u, v, alpha=0.6, linewidth=1, linestyle="--",
                label="noisy" if idx == noisy_ids[0] else "")

    ax.set_aspect("equal", adjustable="box")
    ax.legend()
    fig.tight_layout()
    fig.savefig(save_path, dpi=200)
    plt.close(fig)

# XY
plot_group_projection(
    traj_dict, group_clean_ids, group_noisy_ids,
    plane="xy",
    title="Clean vs Noisy - XY",
    save_path=out_group / "group_clean_vs_noisy_xy.png"
)

# YZ
plot_group_projection(
    traj_dict, group_clean_ids, group_noisy_ids,
    plane="yz",
    title="Clean vs Noisy - YZ",
    save_path=out_group / "group_clean_vs_noisy_yz.png"
)

# XZ
plot_group_projection(
    traj_dict, group_clean_ids, group_noisy_ids,
    plane="xz",
    title="Clean vs Noisy - XZ",
    save_path=out_group / "group_clean_vs_noisy_xz.png"
)

print("그룹 평면 비교 그림 저장 완료.")

In [ ]:
# =========================================================
# 셀 9. [3단계] feature 추출 함수
#   - 중심(평균 위치)
#   - 좌표 공분산(3x3) & 각 축 분산
#   - bounding box 부피
#   - 경로 길이
#   - 중심 기준 반지름 평균 & 표준편차
# =========================================================
def extract_features(xs, ys, zs):
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    zs = np.asarray(zs, dtype=float)

    # 중심 (평균 위치)
    cx, cy, cz = xs.mean(), ys.mean(), zs.mean()

    # 공분산 행렬 (3x3)
    coords = np.vstack([xs, ys, zs])  # shape: (3, N)
    cov = np.cov(coords)  # (3, 3)

    cov_xx, cov_xy, cov_xz = cov[0, 0], cov[0, 1], cov[0, 2]
    cov_yx, cov_yy, cov_yz = cov[1, 0], cov[1, 1], cov[1, 2]
    cov_zx, cov_zy, cov_zz = cov[2, 0], cov[2, 1], cov[2, 2]

    # 각 축 분산 (cov의 대각원소)
    var_x, var_y, var_z = cov_xx, cov_yy, cov_zz

    # bounding box
    x_range = xs.max() - xs.min()
    y_range = ys.max() - ys.min()
    z_range = zs.max() - zs.min()
    bbox_volume = x_range * y_range * z_range

    # path length (시퀀스 순서대로 거리 합)
    if len(xs) >= 2:
        diffs = np.stack([np.diff(xs), np.diff(ys), np.diff(zs)], axis=1)  # (N-1, 3)
        segment_lengths = np.linalg.norm(diffs, axis=1)
        path_length = segment_lengths.sum()
    else:
        path_length = 0.0

    # 중심 기준 반지름
    centered = np.stack([xs - cx, ys - cy, zs - cz], axis=1)  # (N, 3)
    radii = np.linalg.norm(centered, axis=1)
    radius_mean = radii.mean()
    radius_std = radii.std(ddof=1) if len(radii) > 1 else 0.0

    # 특징 dict
    feat = {
        # 중심
        "center_x": cx,
        "center_y": cy,
        "center_z": cz,

        # 분산 (대각)
        "var_x": var_x,
        "var_y": var_y,
        "var_z": var_z,

        # 공분산 (off-diagonal 포함)
        "cov_xy": cov_xy,
        "cov_xz": cov_xz,
        "cov_yz": cov_yz,

        # bounding box
        "x_range": x_range,
        "y_range": y_range,
        "z_range": z_range,
        "bbox_volume": bbox_volume,

        # path length
        "path_length": path_length,

        # 중심 기준 반지름
        "radius_mean": radius_mean,
        "radius_std": radius_std,
    }

    return feat

In [ ]:
# =========================================================
# 셀 10. [3단계] 모든 파일에 대해 feature DataFrame 생성
#   - 가로축: 1~16
#   - 세로축: feature 이름들
# =========================================================
feature_dict_per_file = {}  # key: idx, value: feature dict

for idx in range(1, 17):
    xs, ys, zs = traj_dict[idx]
    feat = extract_features(xs, ys, zs)
    feature_dict_per_file[idx] = feat

# DataFrame으로 변환
# columns = 파일 번호(1~16), index = feature 이름 (세로축)
df_features = pd.DataFrame(feature_dict_per_file)
df_features.index.name = "feature"
df_features.columns.name = "file_id"

display(df_features)

# CSV로 저장해두면 나중에 다시 쓰기 편함
df_features.to_csv(out_features / "circle_features_table.csv")

print("feature 표 생성 및 저장 완료.")

In [ ]:
# =========================================================
# 셀 11. clean vs noisy 그룹 라벨 붙이고 통계 비교
# =========================================================
# 파일별 group 정보 Series 생성
group_labels = {}
for idx in range(1, 17):
    if idx in group_clean_ids:
        group_labels[idx] = "clean"
    elif idx in group_noisy_ids:
        group_labels[idx] = "noisy"
    else:
        group_labels[idx] = "unknown"

group_series = pd.Series(group_labels, name="group")  # index: file_id

# df_features는 (feature x file_id)이므로, 전치해서 (file_id x feature)로 바꾼 후 group join
df_feat_T = df_features.T  # index: file_id, columns: feature
df_feat_T = df_feat_T.join(group_series)

# group 별 평균/표준편차
group_mean = df_feat_T.groupby("group").mean()
group_std = df_feat_T.groupby("group").std()

print("=== 그룹별 feature 평균 ===")
display(group_mean)

print("=== 그룹별 feature 표준편차 ===")
display(group_std)

# 저장
group_mean.to_csv(out_features / "circle_features_group_mean.csv")
group_std.to_csv(out_features / "circle_features_group_std.csv")

In [ ]:
# =========================================================
# 셀 12. [3단계] 박스플롯 & 히스토그램 저장
#   - 몇 가지 핵심 feature에 대해 clean vs noisy 비교
# =========================================================
# 관심 feature들 (원하면 더 추가 가능)
important_features = [
    "bbox_volume",
    "path_length",
    "radius_std",
    "radius_mean",
    "var_x",
    "var_y",
    "var_z",
]

# 박스플롯 (clean vs noisy)
for feat in important_features:
    fig, ax = plt.subplots()
    ax.set_title(f"{feat} - Boxplot (clean vs noisy)")
    data_clean = df_feat_T[df_feat_T["group"] == "clean"][feat]
    data_noisy = df_feat_T[df_feat_T["group"] == "noisy"][feat]
    ax.boxplot(
        [data_clean.values, data_noisy.values],
        labels=["clean", "noisy"],
        showmeans=True,
    )
    ax.set_ylabel(feat)
    fig.tight_layout()
    fig.savefig(out_features / f"boxplot_{feat}.png", dpi=200)
    plt.close(fig)

# 히스토그램 (clean vs noisy를 같은 축에 그리기)
for feat in important_features:
    fig, ax = plt.subplots()
    ax.set_title(f"{feat} - Histogram (clean vs noisy)")
    data_clean = df_feat_T[df_feat_T["group"] == "clean"][feat]
    data_noisy = df_feat_T[df_feat_T["group"] == "noisy"][feat]

    ax.hist(
        data_clean,
        bins=5,
        alpha=0.6,
        label="clean",
    )
    ax.hist(
        data_noisy,
        bins=5,
        alpha=0.6,
        label="noisy",
    )
    ax.set_xlabel(feat)
    ax.set_ylabel("Count")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_features / f"hist_{feat}.png", dpi=200)
    plt.close(fig)

print("박스플롯 & 히스토그램 저장 완료.")